# Chapter facts: the figures behind the new chapter openings

This notebook computes the new figures for the single-page story: how close copper and four other commodities are to their own records (the interlude
"Is it just copper?"), whether August 2026 is also a record in euros (chapter 1), mine production and reserves by country with a simple reserve-life
ratio (chapter 4), and checks on the price records named in the events table. Each figure is written to `results/` and checked here.

**Read this first.** These are descriptions of the data. Nothing here explains why prices moved, and nothing is a forecast. Reserve life is a simple
ratio of reserves to one year of mine output, not a forecast of when a country runs out.

Run `python copper_database/build_db.py --raw .` first.

In [1]:
import sqlite3
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
con = sqlite3.connect("../copper_database/copper.db")
rd = lambda sql: pd.read_sql(sql, con)
written = {}


def save(df, name):
    df.to_csv(f"../results/{name}.csv", index=False)
    df.to_sql(name, con, if_exists="replace", index=False)
    written[name] = (len(df), len(df.columns))

## 1. Is it just copper? Each commodity against its own record
World Bank monthly prices (nominal US dollars, monthly averages, source S02), in today's money with US CPI (BLS via FRED, S04), August 2026 dollars.
"Its own record" means the highest month since that series starts. All five series start in January 1960, so the comparison covers the same span.
US CPI has no value for October 2025 (S19), so that month has no real price; it is not a record month for any series, so the result does not depend on it.
Gold: central banks held the price at about $35 an ounce until the London Gold Pool collapsed in March 1968 (S27); the series shows exactly $35 until then,
so its early years are not a free market price. Brent and aluminium: the Pink Sheet description sheet says what each series is from recent years on, but not
what the earliest years are; the data itself shows that Brent, Dubai and the oil average are the same number every month until December 1978.

In [2]:
COMMODITIES = {"Copper": "copper", "Gold": "gold", "Aluminum": "aluminium", "Crude oil, Brent": "brent", "Tin": "tin"}
wb = rd("SELECT date, commodity, value FROM stg_wb_prices_monthly WHERE commodity IN ('" + "','".join(COMMODITIES) + "')")
wide = wb.pivot(index="date", columns="commodity", values="value").sort_index()
cpi = rd("SELECT date, value FROM stg_fred_monthly WHERE series_id = 'CPIAUCSL'").set_index("date")["value"]
last = wide.index.max()
assert last == "2026-08-01" and not np.isnan(cpi.loc[last]), "the latest month must have a CPI value"
real = wide.mul(cpi.loc[last] / cpi, axis=0)        # NaN where CPI is missing (October 2025)

rows = []
for wbname, key in COMMODITIES.items():
    n, r = wide[wbname].dropna(), real[wbname].dropna()
    rows.append({
        "commodity": key, "world_bank_name": wbname, "series_start": n.index.min()[:7], "latest_month": last[:7],
        "latest_nominal": n.loc[last], "nominal_peak_month": n.idxmax()[:7], "nominal_peak": n.max(), "latest_pct_of_nominal_peak": n.loc[last] / n.max() * 100,
        "latest_real": r.loc[last], "real_peak_month": r.idxmax()[:7], "real_peak": r.max(), "latest_pct_of_real_peak": r.loc[last] / r.max() * 100,
        "at_nominal_record": bool(n.loc[last] >= n.max()), "at_real_record": bool(r.loc[last] >= r.max()),
    })
rec = pd.DataFrame(rows)
# checks: copper agrees with the Story facts; gold fixed price in the 1960s; no commodity's real peak sits in the month without CPI
facts = rd("SELECT fact_id, month, value FROM res_story_copper_record_facts").set_index("fact_id")
cu = rec.set_index("commodity").loc["copper"]
assert cu.real_peak_month == facts.loc["real_peak_all", "month"][:7]
assert abs((cu.real_peak / cu.latest_real - 1) * 100 - facts.loc["real_peak_vs_latest_pct", "value"]) < 0.01
assert cu.at_nominal_record and not cu.at_real_record
assert (wide.loc["1960-01-01":"1967-12-01", "Gold"] == 35).all(), "gold should be fixed at $35 in 1960 to 1967"
first_free_gold = wide.index[(wide.Gold != 35)].min()
assert first_free_gold[:4] == "1968", "gold should leave $35 in 1968"
oil = rd("SELECT date, commodity, value FROM stg_wb_prices_monthly WHERE commodity IN ('Crude oil, Brent', 'Crude oil, Dubai', 'Crude oil, average')").pivot(index="date", columns="commodity", values="value")
same = (oil["Crude oil, Brent"] == oil["Crude oil, Dubai"]) & (oil["Crude oil, Brent"] == oil["Crude oil, average"])
last_same = same[same].index.max()
assert same.loc[:last_same].all() and not same.loc[last_same:].iloc[1:].all(), "Brent should equal Dubai and the average without a break until one month"
assert last_same == "1978-12-01"
assert "2025-10" not in set(rec.real_peak_month)
assert rec.series_start.eq("1960-01").all()
rec["note_kind"] = ""
rec["note_until"] = ""
rec["note_source"] = ""
rec.loc[rec.commodity == "gold", ["note_kind", "note_until", "note_source"]] = ["gold_fixed", first_free_gold[:4], "S27"]
rec.loc[rec.commodity == "brent", ["note_kind", "note_until", "note_source"]] = ["brent_same", last_same[:7], "S02"]
rec.loc[rec.commodity == "aluminium", ["note_kind", "note_until", "note_source"]] = ["aluminium_undescribed", "", "S02"]
for c in ["latest_nominal", "nominal_peak", "latest_real", "real_peak"]:
    rec[c] = rec[c].round(2)
for c in ["latest_pct_of_nominal_peak", "latest_pct_of_real_peak"]:
    rec[c] = rec[c].round(1)
for c in ["at_nominal_record", "at_real_record"]:
    rec[c] = rec[c].astype(int)        # 1 or 0, so the page can test them directly
save(rec, "res_interlude_records")
rec[["commodity", "series_start", "nominal_peak_month", "latest_pct_of_nominal_peak", "real_peak_month", "latest_pct_of_real_peak", "at_nominal_record", "at_real_record"]]

,commodity,series_start,nominal_peak_month,latest_pct_of_nominal_peak,real_peak_month,latest_pct_of_real_peak,at_nominal_record,at_real_record
0,copper,1960-01,2026-08,100.0,1974-04,68.0,1,0
1,gold,1960-01,2026-02,87.9,2026-02,86.1,0,0
2,aluminium,1960-01,2026-05,88.7,1988-06,32.1,0,0
3,brent,1960-01,2008-07,67.9,2008-06,44.5,0,0
4,tin,1960-01,2026-08,100.0,1978-10,72.9,1,0


## 2. Is August 2026 also a record in euros?
Copper in euros per tonne: the World Bank dollar price divided by the dollars per euro rate (FRED EXUSEU, S16), both monthly averages, from the panel.
The euro series starts in January 1999, so "record" means the highest since then.

In [3]:
eu = rd("SELECT month, copper_eur_t_avg FROM mart_monthly_panel WHERE copper_eur_t_avg IS NOT NULL ORDER BY month").set_index("month")["copper_eur_t_avg"]
prev = eu.iloc[:-1]
euro = pd.DataFrame([
    ("eur_series_start", eu.index.min()[:7], len(eu), "months", "first month with a euro price"),
    ("eur_latest", eu.index.max()[:7], round(eu.iloc[-1], 0), "EUR per tonne", "latest monthly average"),
    ("eur_previous_high", prev.idxmax()[:7], round(prev.max(), 0), "EUR per tonne", "highest month before the latest"),
    ("eur_latest_is_record", eu.index.max()[:7], int(eu.iloc[-1] >= eu.max()), "1 = yes", "the latest month is the highest since the euro series starts"),
    ("eur_latest_vs_previous_high_pct", eu.index.max()[:7], round((eu.iloc[-1] / prev.max() - 1) * 100, 1), "percent", "how far the latest month is above the previous high"),
], columns=["fact_id", "month", "value", "unit", "note"])
assert euro.set_index("fact_id").loc["eur_latest_is_record", "value"] == 1
save(euro, "res_euro_record")
euro

,fact_id,month,value,unit,note
0,eur_series_start,1999-01,332.0,months,first month with a euro price
1,eur_latest,2026-08,12356.0,EUR per tonne,latest monthly average
2,eur_previous_high,2026-07,11856.0,EUR per tonne,highest month before the latest
3,eur_latest_is_record,2026-08,1.0,1 = yes,the latest month is the highest since the euro...
4,eur_latest_vs_previous_high_pct,2026-08,4.2,percent,how far the latest month is above the previous...


## 3. Who supplies it? Mine production and reserves by country
USGS Mineral Commodity Summaries 2026 (S06): mine production for 2025 (estimates) and reserves, in thousand tonnes of copper content.
Shares are of the USGS world total. Reserve life is reserves divided by 2025 mine production: a simple ratio at today's rate, not a forecast of depletion.
"Other countries" is a USGS total for countries not listed one by one; it cannot be placed on a map. The site writes "DR Congo" for the USGS name "Congo (Kinshasa)".

In [4]:
prod = rd("SELECT country, value AS production_2025e_kt, is_estimate FROM stg_usgs_copper WHERE statistic_detail = 'Mine production' AND year = 2025 AND value IS NOT NULL")
res = rd("SELECT country, value AS reserves_kt FROM stg_usgs_copper WHERE statistic_detail = 'Reserves' AND value IS NOT NULL")
iso = rd("SELECT usgs_name AS country, display_name, iso_a3, m49, placeable FROM usgs_country_iso")
world = rd("SELECT statistic, year, value FROM stg_usgs_copper WHERE country LIKE 'World%' AND section LIKE 'World Mine%'")
world_prod = world[(world.statistic == "Production") & (world.year == 2025)].value.min()     # the smaller 2025 'Production' row is mine output (the other is refinery)
world_res = world[world.statistic == "Reserves"].value.iloc[0]
sup = prod.merge(res, on="country", how="outer").merge(iso, on="country", how="left")
assert sup.display_name.notna().all(), "a USGS country has no ISO mapping"
assert prod.is_estimate.eq(1).all(), "2025 production should be marked as estimates"
assert abs(prod.production_2025e_kt.sum() - world_prod) / world_prod < 0.005, "countries plus other countries should add up to the world total"
assert abs(res.reserves_kt.sum() - world_res) / world_res < 0.005
sup["share_of_world_production_pct"] = (sup.production_2025e_kt / world_prod * 100).round(1)
sup["share_of_world_reserves_pct"] = (sup.reserves_kt / world_res * 100).round(1)
sup["reserve_life_years"] = (sup.reserves_kt / sup.production_2025e_kt).round(0)
sup["m49"] = sup.m49.map(lambda v: "" if pd.isna(v) else "%03d" % int(v))
sup = sup.sort_values(["placeable", "production_2025e_kt"], ascending=[False, False]).reset_index(drop=True)
sup["rank"] = np.where(sup.placeable == 1, range(1, len(sup) + 1), np.nan)
world_row = pd.DataFrame([{"country": "World total", "display_name": "World", "production_2025e_kt": world_prod, "reserves_kt": world_res, "is_estimate": 1,
                           "placeable": 0, "share_of_world_production_pct": 100.0, "share_of_world_reserves_pct": 100.0,
                           "reserve_life_years": round(world_res / world_prod, 0), "iso_a3": "", "m49": "", "rank": np.nan}])
sup = pd.concat([sup, world_row], ignore_index=True)
cols = ["rank", "country", "display_name", "iso_a3", "m49", "placeable", "production_2025e_kt", "is_estimate", "share_of_world_production_pct",
        "reserves_kt", "share_of_world_reserves_pct", "reserve_life_years"]
save(sup[cols], "res_supply_countries")
top2 = sup.iloc[:2]
print("Top two:", ", ".join(top2.display_name), "=", round(top2.production_2025e_kt.sum() / world_prod * 100, 1), "% of world mine output (2025 estimates)")
sup[cols]

Top two: Chile, DR Congo = 37.0 % of world mine output (2025 estimates)


,rank,country,display_name,iso_a3,m49,placeable,production_2025e_kt,is_estimate,share_of_world_production_pct,reserves_kt,share_of_world_reserves_pct,reserve_life_years
0,1.0,Chile,Chile,CHL,152,1,5300.0,1,23.0,180000.0,18.4,34.0
1,2.0,Congo (Kinshasa),DR Congo,COD,180,1,3200.0,1,13.9,80000.0,8.2,25.0
2,3.0,Peru,Peru,PER,604,1,2700.0,1,11.7,85000.0,8.7,31.0
3,4.0,China,China,CHN,156,1,1800.0,1,7.8,41000.0,4.2,23.0
4,5.0,Russia,Russia,RUS,643,1,1300.0,1,5.7,80000.0,8.2,62.0
5,6.0,United States,United States,USA,840,1,1000.0,1,4.3,47000.0,4.8,47.0
6,7.0,Zambia,Zambia,ZMB,894,1,940.0,1,4.1,21000.0,2.1,22.0
7,8.0,Australia,Australia,AUS,036,1,730.0,1,3.2,100000.0,10.2,137.0
8,9.0,Indonesia,Indonesia,IDN,360,1,710.0,1,3.1,21000.0,2.1,30.0
9,10.0,Kazakhstan,Kazakhstan,KAZ,398,1,710.0,1,3.1,20000.0,2.0,28.0


## 4. The records named in the events table
Three markers on the chapter 1 chart are records in our own data (source S02). Check that each statement is true.

In [5]:
ser = rd("SELECT month, nominal_usd_t FROM res_story_copper_series ORDER BY month").set_index("month")["nominal_usd_t"]
ev = rd("SELECT event_id, month, label, description, kind, status, source_id FROM events").set_index("event_id")
feb11 = ser.loc["2011-02-01"]
assert feb11 == ser.loc[:"2011-02-01"].max(), "February 2011 should be the highest month as quoted up to then"
first_above = ser.loc["2011-03-01":][ser.loc["2011-03-01":] > feb11].index.min()
assert first_above[:7] == "2021-05", "the 2011 high should first be passed in May 2021 (the E3 description says it holds until then)"
assert ser.idxmax()[:7] == "2026-08" == ev.loc["E8", "month"] and ev.loc["E3", "month"] == "2011-02"
checks = pd.DataFrame([
    ("E3", "February 2011 is the highest month as quoted up to then; it holds until " + first_above[:7], 1),
    ("E8", "August 2026 is the highest month as quoted since 1960", 1),
], columns=["event_id", "statement", "holds"])
save(checks, "res_event_record_checks")

# the approved events with their source, for the chart; each month must be on the price series
src = rd("SELECT source_id, name AS source_name, url AS source_url FROM sources").set_index("source_id")
approved = ev[ev.status == "approved"].reset_index().sort_values("month")
assert all((m + "-01") in ser.index for m in approved.month), "an event month is not on the price series"
approved = approved.join(src, on="source_id")
assert approved.source_url.notna().all()
approved["nominal_usd_t"] = [ser.loc[m + "-01"] for m in approved.month]
save(approved[["event_id", "month", "label", "description", "kind", "source_id", "source_name", "source_url", "nominal_usd_t"]], "res_events")
approved[["event_id", "month", "label", "kind", "source_id"]]

,event_id,month,label,kind,source_id
0,E1,1973-10,Oil embargo begins,event,S22
1,E2,2008-09,Lehman Brothers fails,event,S23
2,E3,2011-02,Record as quoted,record,S02
3,E4,2020-03,Pandemic declared,event,S24
4,E7,2025-09,Grasberg mine accident,event,S26
5,E8,2026-08,New record as quoted,record,S02


## 5. Dollar chapter: each month as a point (monthly % changes)
The dollar chapter's main chart puts every month from February 2006 on a scatter: the broad dollar index's monthly % change across, copper's down.
Charts use simple % changes (the statistics use log changes). The fitted line is the same regression as model 2 in notebook 01 (log changes), so its slope
is the "about 2.8% lower" figure; on the chart it is drawn in % terms. The month with no change in copper counts in neither direction.

In [6]:
ch = rd("""SELECT month, dollar_index_pct_m AS dollar_pct, copper_usd_pct_m AS copper_pct, dollar_index_logret_m AS dollar_log, copper_usd_logret_m AS copper_log
           FROM mart_monthly_changes WHERE dollar_index_pct_m IS NOT NULL AND copper_usd_pct_m IS NOT NULL ORDER BY month""")
ch["direction"] = np.select([(ch.copper_pct > 0) & (ch.dollar_pct < 0), (ch.copper_pct < 0) & (ch.dollar_pct > 0), (ch.copper_pct == 0) | (ch.dollar_pct == 0)],
                            ["opposite", "opposite", "unchanged"], "same")
facts_g = rd("SELECT fact_id, value FROM res_story_guess_dollar").set_index("fact_id")["value"]
assert (ch.direction == "opposite").sum() == facts_g["opposite_months"] and (ch.direction == "unchanged").sum() == facts_g["unchanged_months"] and len(ch) == facts_g["months_total"]
slope, intercept = np.polyfit(ch.dollar_log, ch.copper_log, 1)
model2 = rd("SELECT coefficient FROM res_dollar_regressions WHERE model LIKE '2.%' AND variable = 'dollar_index_logret_m'").coefficient.iloc[0]
assert abs(slope - model2) < 1e-6, "the fitted line must be the regression shown on the page"
sc = ch[["month", "dollar_pct", "copper_pct", "direction"]].copy()
sc["dollar_pct"] = sc.dollar_pct.round(3)
sc["copper_pct"] = sc.copper_pct.round(3)
save(sc, "res_dollar_scatter")
fit = pd.DataFrame([("slope_log", slope, "copper log change per unit dollar log change (model 2)"), ("intercept_log", intercept, "intercept of the same regression"),
                    ("opposite_months", int((ch.direction == "opposite").sum()), "months in the two opposite-direction quadrants"),
                    ("months", len(ch), "months on the chart"), ("unchanged_month", ch.loc[ch.direction == "unchanged", "month"].iloc[0][:7], "the month with no change in copper")],
                   columns=["fact_id", "value", "note"])
save(fit, "res_dollar_scatter_fit")
fit

,fact_id,value,note
0,slope_log,-2.770851,copper log change per unit dollar log change (...
1,intercept_log,0.00642,intercept of the same regression
2,opposite_months,178,months in the two opposite-direction quadrants
3,months,247,months on the chart
4,unchanged_month,2018-09,the month with no change in copper


In [7]:
con.commit()
con.close()
print("Written (rows, columns):")
for k, v in written.items():
    print(f"  {k}: {v[0]} rows, {v[1]} columns")

Written (rows, columns):
  res_interlude_records: 5 rows, 17 columns
  res_euro_record: 5 rows, 5 columns
  res_supply_countries: 16 rows, 12 columns
  res_event_record_checks: 2 rows, 3 columns
  res_events: 6 rows, 9 columns
  res_dollar_scatter: 247 rows, 4 columns
  res_dollar_scatter_fit: 5 rows, 3 columns
